# Intro to NumPy

A hands-on companion to class notes on NumPy.

## 0. Setup

NumPy is not in the Python standard library, so install it once per environment.

```bash
conda activate <your-env>   # or `conda create -n env python=3.x numpy`
conda install numpy
```

Then verify the install and version.

In [1]:
import numpy as np

print("numpy version:", np.__version__)

numpy version: 2.5.3


## 1. The Memory Problem - Why Python lists are slow

### 1.1 What a Python list actually is

A Python list is an **array of pointers** (references). Each element lives
anywhere on the **heap**, is a **boxed Python object**, and can even be a
different type:

```
say one int = 4 bytes

Stack:
  x = 0x447   # x holds a *reference* (memory address), not the data itself

Heap (allocated at runtime, accessed through the reference above):
  address  contents
  0x447    --> points to scattered objects on the heap
              x = [1, 'Hello', ..15, [], 1234]   # heterogeneous!  any type
              each element is a separate pointer to a boxed object elsewhere
```

That flexibility is **expensive**:
- Each value needs its own pointer + Python object header (~28 bytes/object).
- Reading `x[3]` means **following a pointer** to another memory location.
- The CPU cache loves **contiguous** data, not scattered pointers.

### 1.2 Visualising the "mess"

```
int x = new int[10]        # Java/C# analogy - Python lists = list-of-pointers-to-boxed-objects

x = 0x447 ----------------- Heap
                      +---------+ 4 bytes
  0x447 --> index 0   |         |    --> int(1) object (28 bytes)
                      +---------+
  index 1             |         |    --> str 'Hello' object
                      +---------+
  index 2             |         |    --> ...
                      +---------+
  index 3             |         |    --> int(4) ... wait, x[3] follows a pointer
                      +---------+
  index 4             |         |
                      +---------+
```

### 1.3 NumPy's fix - contiguous, homogeneous arrays

```
import numpy as np
x = np.array([1, 2, 3, 4])   # every element is the SAME dtype, laid out back-to-back

  x[3]

                In Memory (one contiguous block - no pointer chasing)
                   x = 0x447 --> +---------+ 0  (1)
                                  +---------+ 1  (2)
                                  +---------+ 2  (3)
                                  +---------+ 3  (4)
                                  +---------+
```

Key consequence: **`x[3]` reads memory at `base + 3*itemsize` directly** -
one address arithmetic, no dereferencing of a second pointer.

## 2. Creating arrays

An ndarray is **homogeneous**: pick one `dtype` and every element matches.

- `np.array(...)` infers the dtype from Python data.
- `np.zeros` / `np.ones` / `np.empty` preallocate (much faster than appending to a list).
- `np.arange` is the `range()` for arrays; `np.linspace` gives evenly-spaced samples.

In [2]:
# From Python lists
a = np.array([1, 2, 3, 4])        # int64 by default
b = np.array([1.0, 2.0, 3.0])     # float64
c = np.array([[1, 2], [3, 4]])   # 2-D

# Preallocated (use these instead of growing a list)
z = np.zeros(5)                    # [0., 0., 0., 0., 0.]
o = np.ones((2, 3))              # 2x3 of ones
e = np.empty(3)                  # 3 slots, contents uninitialized (garbage)

# Ranges and sequences
r = np.arange(0, 10, 2)          # [0, 2, 4, 6, 8]
l = np.linspace(0, 1, 6)         # [0. , .2, .4, .6, .8, 1.]

print("a:", a, "| dtype:", a.dtype)
print("b:", b, "| dtype:", b.dtype)
print("c (2-D):"); print(c)
print("z:", z, "| o shape:", o.shape, "| r:", r, "| l:", l)

a: [1 2 3 4] | dtype: int64
b: [1. 2. 3.] | dtype: float64
c (2-D):
[[1 2]
 [3 4]]
z: [0. 0. 0. 0. 0.] | o shape: (2, 3) | r: [0 2 4 6 8] | l: [0.  0.2 0.4 0.6 0.8 1. ]


## 3. dtypes - the heart of NumPy's speed

Because every element is the same size, NumPy is a **C** array under the hood.

Common dtypes:

| dtype | meaning | bytes |
|-------|---------|-------|
| `int8`  | signed byte     | 1 |
| `int32` | 32-bit int      | 4 |
| `int64` | 64-bit int      | 8 |
| `uint8` | unsigned byte   | 1 |
| `float32` | 32-bit float  | 4 |
| `float64` | 64-bit float (default) | 8 |
| `bool`  | True/False      | 1 |

In [3]:
print("default int dtype:", np.array([1, 2]).dtype)
print("forced float32:", np.array([1, 2], dtype=np.float32).dtype)

small = np.array([1, 2, 3], dtype=np.int8)   # uses 3 bytes total!
print("int8 nbytes:", small.nbytes, " vs int64 nbytes:", np.array([1, 2, 3]).nbytes)

big = np.array([1, 2, 3, 4], dtype=np.float64).astype(np.float32)
print("casted:", big, big.dtype)

default int dtype: int64
forced float32: float32
int8 nbytes: 3  vs int64 nbytes: 24
casted: [1. 2. 3. 4.] float32


## 4. Array attributes

Every array carries metadata describing its shape in memory:

- `ndim` - number of axes (dimensions).
- `shape` - tuple of length `ndim`, the size along each axis.
- `size` - total number of elements (`prod(shape)`).
- `dtype` - element type.
- `itemsize` - bytes per element.
- `nbytes` - total bytes consumed (`size * itemsize`).
- `T` / `.transpose()` - swap the axes (view, no copy).

In [4]:
m = np.arange(24).reshape(2, 3, 4)
print("ndim   :", m.ndim)
print("shape  :", m.shape)
print("size   :", m.size)
print("dtype  :", m.dtype)
print("itemsize:", m.itemsize)
print("nbytes :", m.nbytes, " (== size*itemsize:", m.size * m.itemsize, ")")
print("T shape:", m.T.shape)

ndim   : 3
shape  : (2, 3, 4)
size   : 24
dtype  : int64
itemsize: 8
nbytes : 192  (== size*itemsize: 192 )
T shape: (4, 3, 2)


## 5. Indexing & slicing

NumPy mirrors Python's slice syntax but extends it to **multiple dimensions**.

- **Basic indexing**: `a[2]`, `a[1:4]`, `a[::-1]`.
- **Multidimensional**: `m[row, col]`, `m[:, 0]` (all rows, col 0).
- **Boolean mask**: `m[m > 5]` keeps only elements where the mask is `True`.
- **Integer-array (fancy) indexing**: `m[[0, 2]]` picks rows 0 and 2.

Warning: Slices **return views** (see section 8). A write to the slice modifies the original.

In [5]:
v = np.arange(10)
print("v[2]      =", v[2])
print("v[1:5]    =", v[1:5])
print("v[::-1]   =", v[::-1], "  (reverse, strided view)")

m = np.arange(12).reshape(3, 4)
print("\nm="); print(m)
print("row 1  =", m[1, :])        # row 1, all cols
print("col 2  =", m[:, 2])        # all rows, col 2
print(">5 mask:", m[m > 5].ravel())   # flatten result
print("rows [0,2]:"); print(m[[0, 2]])

v[2]      = 2
v[1:5]    = [1 2 3 4]
v[::-1]   = [9 8 7 6 5 4 3 2 1 0]   (reverse, strided view)

m=
[[ 0  1  2  3]
 [ 4  5  6  7]
 [ 8  9 10 11]]
row 1  = [4 5 6 7]
col 2  = [ 2  6 10]
>5 mask: [ 6  7  8  9 10 11]
rows [0,2]:
[[ 0  1  2  3]
 [ 8  9 10 11]]


## 6. Vectorization - stop writing Python `for` loops

The single biggest win with NumPy: operations happen in **compiled C** over the
whole array at once. Compare these two ways to add 1 to every element.

In [6]:
import time

data = np.arange(1_000_000)

# --- slow: Python loop, box/unbox every int ---
slow = []
t0 = time.perf_counter()
for x in data:
    slow.append(x + 1)
t1 = time.perf_counter()
print(f"Python loop : {t1 - t0:.4f}s")

# --- fast: vectorized, runs in C ---
t0 = time.perf_counter()
fast = data + 1
t1 = time.perf_counter()
print(f"NumPy + 1   : {t1 - t0:.4f}s")

print("equal?:", np.array_equal(slow, fast), "| sample fast[:3]:", fast[:3])

Python loop : 0.0822s
NumPy + 1   : 0.0015s
equal?: True | sample fast[:3]: [1 2 3]


## 7. Broadcasting - arithmetic on arrays of different shapes

NumPy *broadcasts* a smaller array across a larger one so the operation still
makes sense. Two rules of thumb:

1. Align on the **rightmost** axis.
2. A dimension of size 1 is **stretched** to match the other.

Example: add a 1-D vector of length 3 to every row of a 2x3 matrix.

In [7]:
matrix = np.array([[1, 2, 3],
                    [4, 5, 6]])
vector = np.array([10, 20, 30])

result = matrix + vector     # vector broadcast across each ROW
print(result)
print("shape:", result.shape)

# Column-wise broadcast: make the vector shape (3,1) and add to a (3,2)
col = np.array([[10], [20], [30]])   # shape (3,1)
mat2 = np.array([[1, 2], [3, 4], [5, 6]])
print("\ncol + mat2:"); print(col + mat2)

[[11 22 33]
 [14 25 36]]
shape: (2, 3)

col + mat2:
[[11 12]
 [23 24]
 [35 36]]


## 8. Views vs copies - when does data actually get duplicated?

Most operations return a **view** (a window onto the same underlying buffer).
That is free and fast, but it means *modifying a view modifies the original*.
Use `.copy()` whenever you need an independent array.

In [8]:
base = np.arange(6)
view = base[1:4]          # a view, shares memory
view[0] = 99              # mutates `base`!
print("after mutate view:", base)   # base[1] is now 99

copy = base[1:4].copy()   # independent buffer
copy[0] = -1
print("base unchanged:", base)

# Does my slice own its data?
print("view owns data:", view.flags['OWNDATA'])
print("copy owns data:", copy.flags['OWNDATA'])

after mutate view: [ 0 99  2  3  4  5]
base unchanged: [ 0 99  2  3  4  5]
view owns data: False
copy owns data: True


## 9. Reshaping & stacking

`-1` in a shape means *figure out this dimension for me*. Reshaping is a view
whenever possible (no data copy).

- `a.reshape(new_shape)` - view when shapes are compatible.
- `np.concatenate` / `np.vstack` / `np.hstack` - glue arrays along an axis.
- `a.ravel()` / `a.flatten()` - 1-D flatten (ravel is a view, flatten copies).

In [9]:
flat = np.arange(12)
g = flat.reshape(3, 4)              # 3x4 view
h = flat.reshape(3, -1)             # -1 inferred -> also 3x4
print("3x4 == 3,-1 ?", np.array_equal(g, h))

print(flat.reshape(2, 3, 2).shape)  # works for any compatible split

top    = np.zeros((2, 3))
bottom = np.ones((2, 3))
print("vstack shape:", np.vstack([top, bottom]).shape)   # (4, 3)
print("hstack shape:", np.hstack([top, bottom]).shape)   # (2, 6)

print("ravel vs flatten same?:", np.array_equal(g.ravel(), g.flatten()))

3x4 == 3,-1 ? True
(2, 3, 2)
vstack shape: (4, 3)
hstack shape: (2, 6)
ravel vs flatten same?: True


## 10. Reductions - summarizing along an axis

Methods like `.sum()`, `.mean()`, `.min()`, `.max()` accept an `axis=` argument.
Pick an axis to **collapse**; the operation runs in C over that axis.

In [10]:
m = np.arange(12).reshape(3, 4)
print("m ="); print(m)
print("total sum :", m.sum())
print("row sums  :", m.sum(axis=1))        # collapse each row -> shape (3,)
print("col means :", m.mean(axis=0))        # collapse each col -> shape (4,)
print("col mins  :", m.min(axis=0))
print("(m>5).all per row:", (m > 5).all(axis=1))   # bool reduction

m =
[[ 0  1  2  3]
 [ 4  5  6  7]
 [ 8  9 10 11]]
total sum : 66
row sums  : [ 6 22 38]
col means : [4. 5. 6. 7.]
col mins  : [0 1 2 3]
(m>5).all per row: [False False  True]


## 11. Universal functions (ufuncs)

A **ufunc** is a vectorized function that works element-wise on arrays and
returns a new array. They are the building blocks of fast NumPy code.

Binary ufuncs: `add`, `subtract`, `multiply`, `divide`, `power`, `maximum`.
Unary ufuncs: `sqrt`, `exp`, `log`, `sin`, `cos`, `abs`.

Each ufunc supports the `out=` argument so you can write **in-place** into a
preallocated array - handy to avoid temporary allocations in a hot loop.

In [11]:
x = np.array([1.0, 2.0, 3.0, 4.0])
y = np.array([10.0, 20.0, 30.0, 40.0])

print("add   :", np.add(x, y))
print("power :", np.power(x, 2))
print("sqrt  :", np.sqrt(x))

# in-place: reuse the buffer instead of allocating a new array
np.multiply(x, 2, out=x)
print("x after *=2 in-place:", x)

add   : [11. 22. 33. 44.]
power : [ 1.  4.  9. 16.]
sqrt  : [1.         1.41421356 1.73205081 2.        ]
x after *=2 in-place: [2. 4. 6. 8.]


## 12. Linear algebra & random numbers

NumPy ships `np.linalg` (matrix inverses, determinants, eigenvalues) and
`np.random` (modern: prefer `np.random.default_rng(seed)`).

A **matrix-vector** product `A @ x` is the backbone of ML / scientific code.

In [12]:
A = np.array([[1, 2], [3, 4]])
x = np.array([5, 6])
print("A @ x =", A @ x)                  # matrix-vector product
print("det(A) =", np.linalg.det(A))
print("A^-1 ="); print(np.linalg.inv(A))

rng = np.random.default_rng(42)
samples = rng.normal(loc=0.0, scale=1.0, size=5)
print("normal samples:", np.round(samples, 4))

A @ x = [17 39]
det(A) = -2.0000000000000004
A^-1 =
[[-2.   1. ]
 [ 1.5 -0.5]]
normal samples: [ 0.3047 -1.04    0.7505  0.9406 -1.951 ]
